# 06 — Episodic Memory Agent

**Model:** Gemma 2 (9B) via Nebius AI Studio  
**Pattern:** Short + Long-Term Memory via Vector Store  
**Difficulty:** ⭐⭐⭐☆☆

---

## The Problem

By default, an LLM has no memory between conversations. Every session starts from zero. If a user told the agent their preferences, constraints, or context last week, the agent has no idea.

There are two distinct memory problems:
1. **Within-session memory** (short-term): keeping track of what was said earlier in the current conversation without blowing up the context window.
2. **Cross-session memory** (long-term): recalling facts from *past* conversations when they're relevant today.

This notebook solves both using:
- A sliding-window message buffer for short-term memory
- A Chroma vector store for long-term episodic recall

## Architecture

```
User Input
    │
    ▼
┌──────────────────────────────────────────────────────────┐
│ Retrieve relevant memories from vector store             │
│    ↓                                                     │
│ Build context: [retrieved memories] + [recent messages]  │
│    ↓                                                     │
│ LLM generates response                                   │
│    ↓                                                     │
│ Save important facts from this turn to vector store      │
└──────────────────────────────────────────────────────────┘
    │
    ▼
Response
```

## Setup

In [ ]:
import os
import uuid
from datetime import datetime
from dotenv import load_dotenv
load_dotenv()

from langchain_nebius import ChatNebius
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_chroma import Chroma
from langchain_community.embeddings import FakeEmbeddings  # swap for real embeddings in prod
from langgraph.graph import StateGraph, END
from typing import TypedDict, List, Annotated
import operator

llm = ChatNebius(
    model="google/gemma-2-9b-it",
    temperature=0.3,
    api_key=os.environ["NEBIUS_API_KEY"]
)

print("LLM ready.")

## Memory Store Setup

We use Chroma as an in-memory vector store. In production you'd swap `FakeEmbeddings` for a real embedding model (e.g. `langchain-google-genai` with `text-embedding-004` or a local sentence-transformer).

> **Note:** `FakeEmbeddings` returns random vectors — retrieval won't be semantically meaningful, but the plumbing is identical. Just swap the embedding class for a real one.

In [ ]:
# In production: replace FakeEmbeddings with a real embedding model
embeddings = FakeEmbeddings(size=768)

memory_store = Chroma(
    collection_name="episodic_memory",
    embedding_function=embeddings,
)

print("Memory store initialized.")


def save_memory(content: str, metadata: dict = None) -> None:
    """Save a piece of information to long-term memory."""
    memory_store.add_texts(
        texts=[content],
        metadatas=[{"timestamp": datetime.now().isoformat(), **(metadata or {})}],
        ids=[str(uuid.uuid4())]
    )


def retrieve_memories(query: str, k: int = 3) -> List[str]:
    """Retrieve the k most relevant memories for a given query."""
    docs = memory_store.similarity_search(query, k=k)
    return [doc.page_content for doc in docs]

## State Definition

In [ ]:
class MemoryAgentState(TypedDict):
    messages: Annotated[List, operator.add]   # Full conversation buffer
    user_input: str
    retrieved_memories: List[str]
    response: str
    short_term_window: int  # How many recent messages to include in context

## Graph Nodes

In [ ]:
def retrieve_node(state: MemoryAgentState) -> dict:
    """Pull relevant long-term memories before generating a response."""
    print(f"\n[Memory] Retrieving relevant memories for: '{state['user_input'][:60]}...'")
    memories = retrieve_memories(state["user_input"], k=3)
    if memories:
        print(f"[Memory] Found {len(memories)} relevant memory/memories.")
    else:
        print("[Memory] No relevant memories found.")
    return {"retrieved_memories": memories}


def respond_node(state: MemoryAgentState) -> dict:
    """Generate a response using short-term context + retrieved long-term memories."""
    # Build memory context section
    memory_section = ""
    if state["retrieved_memories"]:
        memory_section = "\n".join(
            f"- {m}" for m in state["retrieved_memories"]
        )
        memory_section = f"\n\nRelevant memories from past conversations:\n{memory_section}"

    system_prompt = (
        "You are a helpful personal assistant with memory. "
        "Use the retrieved memories to personalize your responses. "
        "If memories are relevant to the current question, reference them naturally."
        f"{memory_section}"
    )

    # Short-term: only keep the last N messages
    window = state.get("short_term_window", 6)
    recent_messages = state["messages"][-window:] if state["messages"] else []

    all_messages = [
        SystemMessage(content=system_prompt),
        *recent_messages,
        HumanMessage(content=state["user_input"])
    ]

    response = llm.invoke(all_messages)
    print(f"[Agent] Response: {response.content[:120]}...")
    return {"response": response.content}


def consolidate_node(state: MemoryAgentState) -> dict:
    """
    Extract and save important facts from this turn to long-term memory.
    We ask the LLM to identify facts worth remembering.
    """
    extraction_prompt = (
        "Extract any important personal facts, preferences, or context from this conversation turn "
        "that would be worth remembering for future sessions. "
        "Return ONLY the facts as a bullet list. If nothing notable, say 'Nothing to save.'"
    )

    turn_text = f"User: {state['user_input']}\nAssistant: {state['response']}"

    extraction = llm.invoke([
        SystemMessage(content=extraction_prompt),
        HumanMessage(content=turn_text)
    ])

    facts_text = extraction.content.strip()
    if "nothing" not in facts_text.lower():
        save_memory(facts_text, metadata={"source": "conversation_turn"})
        print(f"[Memory] Saved to long-term memory: {facts_text[:100]}...")
    else:
        print("[Memory] Nothing notable to save.")

    # Append this exchange to the short-term buffer
    return {
        "messages": [
            HumanMessage(content=state["user_input"]),
            AIMessage(content=state["response"])
        ]
    }

## Build the Graph

In [ ]:
builder = StateGraph(MemoryAgentState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("respond", respond_node)
builder.add_node("consolidate", consolidate_node)

builder.set_entry_point("retrieve")
builder.add_edge("retrieve", "respond")
builder.add_edge("respond", "consolidate")
builder.add_edge("consolidate", END)

graph = builder.compile()
print("Memory agent graph compiled.")

## Conversation Helper

We wrap the graph invocation so state is carried between turns.

In [ ]:
# Persistent state across turns
session_state: MemoryAgentState = {
    "messages": [],
    "user_input": "",
    "retrieved_memories": [],
    "response": "",
    "short_term_window": 6
}


def chat(user_message: str) -> str:
    global session_state
    session_state["user_input"] = user_message
    result = graph.invoke(session_state)
    # Update persistent state with new messages
    session_state["messages"] = result["messages"]
    session_state["retrieved_memories"] = []
    return result["response"]

## Live Demo

**Scenario:** A multi-turn conversation where the agent should remember personal preferences introduced early in the session and referenced later.

In [ ]:
# Pre-seed some memories to simulate past sessions
save_memory("User's name is Yehia. He is a software engineer based in Cairo.")
save_memory("User prefers concise, technical explanations without unnecessary filler.")
save_memory("User is learning about LLM agent architectures and building a portfolio on GitHub.")
print("Past memories seeded.")

In [ ]:
turns = [
    "Hi! Can you recommend a good resource for learning LangGraph?",
    "I prefer video tutorials over blog posts, by the way.",
    "Based on what you know about me, what kind of projects should I build next?",
]

for user_msg in turns:
    print(f"\n{'='*60}")
    print(f"USER: {user_msg}")
    print(f"{'='*60}")
    response = chat(user_msg)
    print(f"AGENT: {response}")

## Key Takeaways

| Concept | Implementation |
|---------|---------------|
| **Long-term memory** | Vector store retrieval before every response |
| **Short-term memory** | Sliding window of recent messages in state |
| **Memory consolidation** | LLM extracts saveable facts after each turn |
| **Production upgrade** | Swap `FakeEmbeddings` for a real model; swap in-memory Chroma for persistent storage |

## What's Next

**Notebook 07** explores reasoning breadth — instead of committing to one thought chain, the **Tree of Thoughts** pattern explores multiple parallel paths and selects the best one.